In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
import os
print("当前目录:", os.getcwd())
print("目录内容:", os.listdir("/kaggle/input/competitions"))

当前目录: /kaggle/working
目录内容: ['titanic']


In [3]:
import pandas as pd
import os
# 专业小技巧:动态构造路径以后换别的竞赛也通用
input_dir = "/kaggle/input/competitions/titanic"
train_data = pd.read_csv(os.path.join(input_dir,"train.csv"))
test_data = pd.read_csv(os.path.join(input_dir,"test.csv"))
print("数据加载成功！")
print("训练集大小:", train_data.shape)
print("测试集大小:", test_data.shape)

数据加载成功！
训练集大小: (891, 12)
测试集大小: (418, 11)


In [4]:
print(train_data.head())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


In [5]:
# 按性别统计生存率
print(train_data.groupby('Sex')['Survived'].mean())

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64


In [6]:
# 运行这段代码,看看年龄缺失和舱位的关系
import pandas as pd
# 创建一个"年龄是否缺失"的新列
train_data['Age_Is_Missing'] = train_data['Age'].isna()
# 按舱位统计年龄缺失率
print(train_data.groupby('Pclass')['Age_Is_Missing'].mean())

Pclass
1    0.138889
2    0.059783
3    0.276986
Name: Age_Is_Missing, dtype: float64


In [7]:
import pandas as pd
import os
# 检查数据是否还在
input_dir = "/kaggle/input/competitions/titanic"
train_data = pd.read_csv(os.path.join(input_dir, "train.csv"))
test_data = pd.read_csv(os.path.join(input_dir, "test.csv"))
print("√ 数据加载成功！")
print(f"训练集:{train_data.shape}")
print(f"训练集:{test_data.shape}")
print(train_data.columns.tolist())

√ 数据加载成功！
训练集:(891, 12)
训练集:(418, 11)
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


In [8]:
# 创建副本避免破坏原始数据
train = train_data.copy()
test = test_data.copy()
# 把性别转成数字:male=0,female=1
train['Sex'] = train['Sex'].map({'male': 0, 'female': 1})
test['Sex'] = test['Sex'].map({'male': 0, 'female': 1})
# 处理年龄缺失:用中位数填充
age_median = train['Age'].median()
train['Age'] = train['Age'].fillna(age_median)
test['Age'] = test['Age'].fillna(age_median)
# 处理登船港口缺失:用出现最多的港口填充
embarked_mode = train['Embarked'].mode()[0]
train['Embarked'] = train['Embarked'].fillna(embarked_mode)
test['Embarked'] = test['Embarked'].fillna(embarked_mode)
# 把登船港口也转成数字
train['Embarked'] = train['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
test['Embarked'] = test['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
print("√ 预处理完成！")
print(train[['Sex', 'Age', 'Embarked']].head())


√ 预处理完成！
   Sex   Age  Embarked
0    0  22.0         0
1    1  38.0         1
2    1  26.0         0
3    1  35.0         0
4    0  35.0         0


In [9]:
# 我们选这几个特征
features = ['Pclass', 'Sex', 'Age', 'SibSp','Parch', 'Fare', 'Embarked']
# 从训练集中分出x(特征)和y(目标)
x = train[features]
y = train['Survived']
# 对测试集做同样的处理
x_test = test[features]
print(f"训练特征大小: {x.shape}")
print(f"测试特征大小: {x_test.shape}")
print("选中的特征:", features)

训练特征大小: (891, 7)
测试特征大小: (418, 7)
选中的特征: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
# 创建模型
model = RandomForestClassifier(n_estimators=100, random_state=42)
# 用训练数据"教"它
model.fit(x,y)
# 看看在训练集上的表现(交叉验证)
scores = cross_val_score(model, x, y, cv=5)
print(f"√ 交叉验证准确率: {scores.mean():.4f} (±{scores.std():.4f})")

√ 交叉验证准确率: 0.8115 (±0.0284)


In [11]:
# 对测试集做预测
predictions = model.predict(x_test)
# 生成提交文件
submission = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],
    'Survived': predictions
})
# 保存为CSV
submission.to_csv('submission.csv', index=False)
print("√ 提交文件已生成！")
print(submission.head())

√ 提交文件已生成！
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         1
4          896         0
